# Lab - Temperature and Stop Sequences

Three tasks about two settings that decide *how* Claude answers, not *what* it answers.

| Task | What you learn |
| --- | --- |
| 1 | `temperature=0` - the same question gives the same answer |
| 2 | `temperature=1.0` - the same question gives different answers |
| 3 | `stop_sequences` - generation ends at a marker you choose |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `ANTHROPIC_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

One thing worth knowing before you start: temperature is a dial from `0` to `1.0`.
Near `0` the model almost always picks the most likely next word. Near `1.0` it gives
less likely words a real chance. Neither end is "better" - they suit different jobs.

> **Heads-up (September 2026): the `temperature` setting is being retired.** The `anthropic`
> Python package 1.0+ no longer accepts it (`TypeError: ... unexpected keyword argument
> 'temperature'`), and Claude Sonnet 5 / Opus 4.7+ reject it at the API level. Inside this lab
> nothing changes: the simulator accepts it, and if you take the notebook home,
> `shopassist_lab.py` forwards the value to the real SDK for you. The lesson still holds - low
> temperature for repeatable, policy-driven replies, higher for brainstorming - but on newer
> models you steer consistency with your prompt rather than with this knob.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()

model = "claude-sonnet-4-6"

# The same request is used in all three tasks, so the only thing that
# changes between them is the setting you are testing.
# Note the second line: it asks Claude to finish with a <END> marker,
# which Task 3 uses as a stop sequence.
end_prompt = [
    {
        "role": "user",
        "content": "Write a short customer support reply about a return request.\nEnd the reply with <END>.",
    }
]

---

## Task 1 - Ask the same question three times at temperature 0

In [ ]:
# ============================================================
# TASK 1 - Ask the same question three times at temperature 0
# ============================================================
#
# WHAT TO DO
#   Send the exact same request three times with the
#   temperature set to 0, keep the three replies in a list, and
#   count how many DIFFERENT answers came back.
#
# WHY IT MATTERS
#   At temperature 0 the model takes the most likely next word
#   every time, so the answer stops being a surprise. That is
#   what you want for a refund policy reply, a classification,
#   or anything a customer might screenshot and compare.
#
#   Expect the count below to be 1.
#
# WHERE TO SEE IT IN THE LECTURE
#   "At low temperature, Claude becomes more deterministic. It
#   usually chooses the most likely token" - about 2 minutes
#   22 seconds in.
#
# HOW TO DO IT
#   One blank. The temperature value for the predictable end of
#   the dial - the lecture calls it the low end, and the number
#   is 0.
# ============================================================

deterministic = []

for attempt in range(3):
    # TODO: replace the ... below with the value named beside it
    reply = client.messages.create(
        model=model,
        max_tokens=300,
        temperature=...,   # use 0 - the predictable end of the dial
        messages=end_prompt,
    )
    deterministic.append(reply)

texts = [m.content[0].text for m in deterministic]

print("Sent the same request 3 times at temperature=0.")
print("Distinct replies:", len(set(texts)))
print()
print(texts[0])

check("temperature_zero", deterministic=deterministic)

---

## Task 2 - Ask it three more times at temperature 1.0

In [ ]:
# ============================================================
# TASK 2 - Ask it three more times at temperature 1.0
# ============================================================
#
# WHAT TO DO
#   Exactly the same three requests again. One number changes:
#   the temperature. Then compare the two counts.
#
# WHY IT MATTERS
#   Nothing else about this request is different from Task 1 -
#   same prompt, same model, same max_tokens. Raising the
#   temperature spreads the probability out, so less likely
#   words get a real chance, and the three answers stop
#   matching.
#
#   That is useful for brainstorming campaign ideas. It is a
#   problem for a support reply that has to say the same thing
#   to every customer. Temperature is not good or bad - it is a
#   match to the task.
#
#   Expect a count above 1 this time.
#
# WHERE TO SEE IT IN THE LECTURE
#   "When we move the slider closer to one, the probability
#   becomes more spread out" - about 2 minutes 46 seconds in.
#
# HOW TO DO IT
#   One blank, the same argument as Task 1 with a different
#   value: 1.0 instead of 0.
# ============================================================

varied = []

for attempt in range(3):
    # TODO: replace the ... below with the value named beside it
    reply = client.messages.create(
        model=model,
        max_tokens=300,
        temperature=...,   # use 1.0 this time - the varied end of the dial
        messages=end_prompt,
    )
    varied.append(reply)

print("distinct replies at temperature=0:  ",
      len({m.content[0].text for m in deterministic}))
print("distinct replies at temperature=1.0:",
      len({m.content[0].text for m in varied}))
print()
for i, m in enumerate(varied, 1):
    print("{}. {}".format(i, m.content[0].text.split("\n")[0][:90]))

check("temperature", deterministic=deterministic, varied=varied)

---

## Task 3 - Stop the reply at a marker

In [ ]:
# ============================================================
# TASK 3 - Stop the reply at a marker
# ============================================================
#
# WHAT TO DO
#   Send the request once more, and tell the API to stop
#   generating the moment the text <END> appears. Then read
#   stop_reason and stop_sequence.
#
# WHY IT MATTERS
#   Real support replies are often followed by something the
#   customer must never see - internal notes, a debug section,
#   the next example in a template. A stop sequence cuts the
#   response at a boundary you chose, instead of hoping the
#   model stops on its own.
#
#   Two things to notice in the output. stop_reason is
#   "stop_sequence" and not "end_turn", which is how your code
#   knows WHY it stopped. And the marker itself is not in the
#   text - the API cuts before it.
#
#   A stop sequence controls where generation ends. It is not
#   validation. When you expect JSON or a tool call you still
#   check the result in code, and that is Section 3.
#
# WHERE TO SEE IT IN THE LECTURE
#   "A stop sequence is a custom text marker that tells Claude
#   when to stop generating" - about 4 minutes 3 seconds in.
#
# HOW TO DO IT
#   One blank. The parameter takes a LIST of strings, so the
#   square brackets stay - only the marker goes in the blank,
#   in quotes, exactly as the prompt writes it: "<END>"
# ============================================================

# TODO: replace the ... below with the value named beside it
stopped = client.messages.create(
    model=model,
    max_tokens=300,
    temperature=0,
    stop_sequences=[...],   # the marker "<END>", in quotes, inside the list
    messages=end_prompt,
)

print("stop_reason:  ", stopped.stop_reason)
print("stop_sequence:", stopped.stop_sequence)
print()
print("The reply, cut at the marker:")
print(stopped.content[0].text)
print()
print("Is the marker in the text?", "<END>" in stopped.content[0].text)

check("stop_sequences", stopped=stopped)

---

## Done

Three settings, none of which changed the question you asked:

- `temperature=0` - the same answer every time. The default for support,
  classification and anything policy-driven.
- `temperature=1.0` - room to vary. Good for brainstorming, wrong for a refund rule.
- `stop_sequences` - generation ends at a boundary you named, and `stop_reason` tells
  your code that is why it ended.

That closes the fundamentals. From here the course stops asking Claude for prose and
starts asking it for data your code can rely on - which is where validation, and the
reason stop sequences are not it, becomes the whole topic.